# Phase 2D — Test-Time Augmentation (TTA)

**What this does:** For each image, instead of one forward pass we run 4 augmented versions (original, H-flip, V-flip, both flips) through the model and average the probability vectors. This reduces prediction noise — the model's uncertainty about any single view is smoothed out. No weights are changed. We load the 5 saved Phase 2B checkpoints and re-run inference only.

**Expected improvement:** R2 and R3A are the noisiest classes (fewest training examples). TTA helps most where single-pass variance is highest.

In [1]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import functional as TF
from PIL import Image
import PIL.ImageFile
PIL.ImageFile.LOAD_TRUNCATED_IMAGES = True

import timm
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, cohen_kappa_score, confusion_matrix

sys.path.insert(0, '.')

INPUT_SIZE  = 224
NUM_CLASSES = 4
N_FOLDS     = 5
SEED        = 42
BATCH_SIZE  = 64          # larger than training — no gradients needed
CLASS_NAMES = ['R0', 'R1', 'R2', 'R3A']
OUTPUT_DIR  = Path('output_dir')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Phase 2B checkpoints:', [str(p) for p in sorted((OUTPUT_DIR / 'phase2b_cv').glob('best_fold_*.pth'))])


/home/eth/miniconda3/envs/retfound/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Phase 2B checkpoints: ['output_dir/phase2b_cv/best_fold_0.pth', 'output_dir/phase2b_cv/best_fold_1.pth', 'output_dir/phase2b_cv/best_fold_2.pth', 'output_dir/phase2b_cv/best_fold_3.pth', 'output_dir/phase2b_cv/best_fold_4.pth']


## Load splits and reconstruct fold assignments

In [2]:
GRADE = {'R0': 0, 'R1': 1, 'R2': 2, 'R3A': 3}

df_all = pd.read_csv('labels/splits.csv')
df_all['grade_int'] = df_all['retinopathy'].map(GRADE)

df_cv   = df_all[df_all['split'].isin(['train', 'val'])].copy().reset_index(drop=True)
df_test = df_all[df_all['split'] == 'test'].copy().reset_index(drop=True)

# Reconstruct the exact same folds used in Phase 2B (same StratifiedKFold + SEED=42)
pat_grade = df_cv.groupby('code')['grade_int'].max().reset_index()
pat_grade.columns = ['code', 'strat_grade']
patient_ids   = pat_grade['code'].values
patient_strat = pat_grade['strat_grade'].values

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_assignments = {}
for fold_idx, (_, val_pat_idx) in enumerate(skf.split(patient_ids, patient_strat)):
    for pid in patient_ids[val_pat_idx]:
        fold_assignments[pid] = fold_idx

df_cv['fold'] = df_cv['code'].map(fold_assignments)

print(f'CV images  : {len(df_cv):,}  |  patients: {df_cv["code"].nunique()}')
print(f'Test images: {len(df_test):,}  |  patients: {df_test["code"].nunique()}')
for f in range(N_FOLDS):
    n = (df_cv['fold'] == f).sum()
    print(f'  Fold {f} val: {n} images')

oof_codes  = df_cv['code'].values
test_codes = df_test['code'].values
oof_labels = df_cv['grade_int'].values
test_labels = df_test['grade_int'].values


CV images  : 4,075  |  patients: 990
Test images: 702  |  patients: 175
  Fold 0 val: 818 images
  Fold 1 val: 833 images
  Fold 2 val: 819 images
  Fold 3 val: 804 images
  Fold 4 val: 801 images


## TTA Setup

**4 augmentations** — all deterministic, all valid for circular fundus images:
1. Original (standard eval transform)
2. Horizontal flip
3. Vertical flip
4. Both flips (= 180° rotation)

All augmentations use the same Resize(256)→CenterCrop(224)→Normalize(ImageNet) pipeline that was used during Phase 2B evaluation.

In [3]:
# ── TTA transforms ────────────────────────────────────────────────────────────
_RESIZE = int(INPUT_SIZE / (INPUT_SIZE / 256))   # = 256

def _base_tf(img):
    img = TF.resize(img, _RESIZE, interpolation=TF.InterpolationMode.BICUBIC)
    img = TF.center_crop(img, INPUT_SIZE)
    img = TF.to_tensor(img)
    img = TF.normalize(img, IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
    return img

tta_transforms = [
    lambda img: _base_tf(img),
    lambda img: _base_tf(TF.hflip(img)),
    lambda img: _base_tf(TF.vflip(img)),
    lambda img: _base_tf(TF.vflip(TF.hflip(img))),
]
TTA_NAMES = ['original', 'hflip', 'vflip', 'hflip+vflip']
print(f'{len(tta_transforms)} TTA augmentations defined:', TTA_NAMES)


# ── Image dataset from path list ───────────────────────────────────────────────
class ImageListDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.transform(img)


# ── Model loader (inference-only — no pretrained download needed) ───────────────
def load_model_inference(fold_idx, device):
    """
    Create ViT-Large architecture and load Phase 2B fine-tuned weights.
    pretrained=False avoids downloading the HuggingFace checkpoint;
    we overwrite all weights with our saved state dict anyway.
    """
    model = timm.create_model(
        'vit_large_patch14_dinov2.lvd142m',
        pretrained=False, img_size=INPUT_SIZE,
        num_classes=NUM_CLASSES, drop_path_rate=0.2,
    )
    ckpt_path = OUTPUT_DIR / f'phase2b_cv/best_fold_{fold_idx}.pth'
    state = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(state)
    model.eval()
    return model.to(device)


# ── TTA inference ──────────────────────────────────────────────────────────────
def tta_predict(model, image_paths, transforms, device, batch_size=BATCH_SIZE):
    """
    Run all TTA augmentations over image_paths.
    Returns (N_images, N_classes) averaged across all augmentations.
    """
    aug_probs = []
    for tf in transforms:
        ds = ImageListDataset(image_paths, tf)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                            num_workers=4, pin_memory=True)
        batch_probs = []
        with torch.no_grad(), torch.cuda.amp.autocast():
            for imgs in loader:
                logits = model(imgs.to(device))
                batch_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
        aug_probs.append(np.vstack(batch_probs))
    return np.mean(aug_probs, axis=0)   # average across augmentations


4 TTA augmentations defined: ['original', 'hflip', 'vflip', 'hflip+vflip']


## Run TTA Inference

For each of the 5 Phase 2B fold models:
- Run 4-way TTA on **all 702 test images** (this fold's contribution to the test ensemble)
- Run 4-way TTA on **this fold's ~815 validation images** (OOF contribution)

Then average test probs across all 5 folds (same as the P2B baseline ensemble).

In [4]:
import time

test_paths = df_test['image_path'].values
oof_tta_prbs  = np.zeros((len(df_cv), NUM_CLASSES), dtype=np.float64)
test_fold_prbs = np.zeros((N_FOLDS, len(df_test), NUM_CLASSES), dtype=np.float64)

for fold_idx in range(N_FOLDS):
    t0 = time.time()
    print(f"\n=== Fold {fold_idx} ===")

    model = load_model_inference(fold_idx, device)
    print(f"  Model loaded from best_fold_{fold_idx}.pth")

    # ── Test set: 4-way TTA ────────────────────────────────────────────────
    test_fold_prbs[fold_idx] = tta_predict(model, test_paths, tta_transforms, device)
    print(f"  Test TTA done  ({len(test_paths)} images × {len(tta_transforms)} augs)")

    # ── OOF: this fold's validation images only ────────────────────────────
    val_mask    = df_cv['fold'] == fold_idx
    val_paths   = df_cv.loc[val_mask, 'image_path'].values
    val_indices = df_cv.loc[val_mask].index.values
    oof_tta_prbs[val_indices] = tta_predict(model, val_paths, tta_transforms, device)
    print(f"  OOF TTA done   ({len(val_paths)} images × {len(tta_transforms)} augs)")

    del model
    torch.cuda.empty_cache()
    print(f"  Fold {fold_idx} total: {time.time() - t0:.0f}s")

# Average test probs across 5 fold models (same ensemble logic as P2B baseline)
test_tta_prbs = test_fold_prbs.mean(axis=0)

print("\n✓ TTA inference complete.")
print(f"  OOF TTA shape : {oof_tta_prbs.shape}")
print(f"  Test TTA shape: {test_tta_prbs.shape}")

# Save for future reference
np.save(OUTPUT_DIR / 'phase2b_cv/oof_tta_probs.npy',  oof_tta_prbs)
np.save(OUTPUT_DIR / 'phase2b_cv/test_tta_probs.npy', test_tta_prbs)
print("  Saved to output_dir/phase2b_cv/oof_tta_probs.npy and test_tta_probs.npy")



=== Fold 0 ===


  Model loaded from best_fold_0.pth


/tmp/ipykernel_11435/4284309013.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


  Test TTA done  (702 images × 4 augs)


  OOF TTA done   (818 images × 4 augs)
  Fold 0 total: 288s

=== Fold 1 ===


...

  Test TTA done  (702 images × 4 augs)


  OOF TTA done   (804 images × 4 augs)
  Fold 3 total: 237s

=== Fold 4 ===


  Model loaded from best_fold_4.pth


  Test TTA done  (702 images × 4 augs)


  OOF TTA done   (801 images × 4 augs)
  Fold 4 total: 237s

✓ TTA inference complete.
  OOF TTA shape : (4075, 4)
  Test TTA shape: (702, 4)
  Saved to output_dir/phase2b_cv/oof_tta_probs.npy and test_tta_probs.npy


## Evaluate TTA Results

In [5]:
# ── Helper functions (same as aggregation notebook) ──────────────────────────
def _norm(arr):
    return arr / arr.sum(axis=1, keepdims=True)

def aggregate_by_patient(codes, probs, labels, method='mean'):
    records = {}
    for code, prob, label in zip(codes, probs, labels):
        if code not in records:
            records[code] = {'probs': [], 'worst_grade': 0}
        records[code]['probs'].append(prob)
        records[code]['worst_grade'] = max(records[code]['worst_grade'], int(label))
    patient_codes = sorted(records.keys())
    agg_probs, agg_labels = [], []
    for code in patient_codes:
        stack = np.stack(records[code]['probs'])
        p = stack.mean(axis=0) if method == 'mean' else stack.max(axis=0)
        if method == 'max': p = p / p.sum()
        agg_probs.append(p)
        agg_labels.append(records[code]['worst_grade'])
    return np.array(agg_probs), np.array(agg_labels)

def compute_metrics(labels, probs, preds=None):
    if preds is None:
        preds = probs.argmax(axis=1)
    labels = np.array(labels)
    try:
        auroc = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except Exception:
        auroc = float('nan')
    kappa  = cohen_kappa_score(labels, preds, weights='quadratic')
    cm     = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    sens   = [cm[i, i] / cm[i].sum() if cm[i].sum() > 0 else 0.0 for i in range(NUM_CLASSES)]
    spec   = []
    for i in range(NUM_CLASSES):
        tn = cm.sum() - cm[i].sum() - cm[:, i].sum() + cm[i, i]
        fp = cm[:, i].sum() - cm[i, i]
        spec.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    return {'auroc': auroc, 'kappa': kappa,
            'macro_sens': np.mean(sens), 'macro_spec': np.mean(spec),
            'sensitivity': sens, 'specificity': spec}

def print_metrics(label, m):
    s = m['sensitivity']
    print(f"── {label} ──")
    print(f"  AUROC={m['auroc']:.4f}  Kappa={m['kappa']:.4f}")
    print(f"  Macro sens={m['macro_sens']:.4f}  Macro spec={m['macro_spec']:.4f}")
    for i, cn in enumerate(CLASS_NAMES):
        print(f"  {cn}: sens={s[i]:.4f}  spec={m['specificity'][i]:.4f}")

# ── Normalise ─────────────────────────────────────────────────────────────────
oof_tta_prbs_n  = _norm(oof_tta_prbs)
test_tta_prbs_n = _norm(test_tta_prbs)

# ── Load baseline P2B probs for comparison ────────────────────────────────────
p2b_oof_prb = _norm(np.load(OUTPUT_DIR / 'phase2b_cv/oof_probs_all.npy').astype(np.float64))
p2b_tst_prb = _norm(np.load(OUTPUT_DIR / 'phase2b_cv/test_ensemble_probs.npy').astype(np.float64))
p2b_oof_lbl = np.load(OUTPUT_DIR / 'phase2b_cv/oof_labels_all.npy').astype(int)
p2b_tst_lbl = np.load(OUTPUT_DIR / 'phase2b_cv/test_ensemble_labels.npy').astype(int)

# ── Patient aggregation — TTA ─────────────────────────────────────────────────
pt_tta_mean_prb, pt_tta_lbl = aggregate_by_patient(test_codes, test_tta_prbs_n, test_labels, 'mean')
pt_tta_max_prb,  _           = aggregate_by_patient(test_codes, test_tta_prbs_n, test_labels, 'max')

# ── Patient aggregation — P2B baseline ───────────────────────────────────────
pt_p2b_mean_prb, pt_p2b_lbl = aggregate_by_patient(test_codes, p2b_tst_prb, p2b_tst_lbl, 'mean')
pt_p2b_max_prb,  _           = aggregate_by_patient(test_codes, p2b_tst_prb, p2b_tst_lbl, 'max')

# ── Compute all metrics ───────────────────────────────────────────────────────
m_p2b_img       = compute_metrics(p2b_tst_lbl, p2b_tst_prb)
m_tta_img       = compute_metrics(test_labels,  test_tta_prbs_n)
m_p2b_pt_mean   = compute_metrics(pt_p2b_lbl,  pt_p2b_mean_prb)
m_tta_pt_mean   = compute_metrics(pt_tta_lbl,   pt_tta_mean_prb)
m_p2b_pt_max    = compute_metrics(pt_p2b_lbl,  pt_p2b_max_prb)
m_tta_pt_max    = compute_metrics(pt_tta_lbl,   pt_tta_max_prb)

print_metrics("P2B  | Image  | Argmax  (baseline)",   m_p2b_img)
print()
print_metrics("P2B  | Image  | TTA + Argmax",         m_tta_img)
print()
print_metrics("P2B  | PtMean | Argmax  (best so far)", m_p2b_pt_mean)
print()
print_metrics("P2B  | PtMean | TTA + Argmax",          m_tta_pt_mean)
print()
print_metrics("P2B  | PtMax  | Argmax  (baseline)",    m_p2b_pt_max)
print()
print_metrics("P2B  | PtMax  | TTA + Argmax",          m_tta_pt_max)


── P2B  | Image  | Argmax  (baseline) ──
  AUROC=0.9271  Kappa=0.7671
  Macro sens=0.6375  Macro spec=0.9259
  R0: sens=0.9463  spec=0.8424
  R1: sens=0.7458  spec=0.8884
  R2: sens=0.6078  spec=0.9800
  R3A: sens=0.2500  spec=0.9926

── P2B  | Image  | TTA + Argmax ──
  AUROC=0.9294  Kappa=0.7606
  Macro sens=0.6410  Macro spec=0.9251
  R0: sens=0.9361  spec=0.8553
  R1: sens=0.7458  spec=0.8798
  R2: sens=0.5490  spec=0.9754
  R3A: sens=0.3333  spec=0.9897

── P2B  | PtMean | Argmax  (best so far) ──
  AUROC=0.9456  Kappa=0.8212
  Macro sens=0.6788  Macro spec=0.9389
  R0: sens=0.9890  spec=0.8571
  R1: sens=0.8095  spec=0.9107
  R2: sens=0.5833  spec=0.9877
  R3A: sens=0.3333  spec=1.0000

── P2B  | PtMean | TTA + Argmax ──
  AUROC=0.9450  Kappa=0.8131
  Macro sens=0.6709  Macro spec=0.9344
  R0: sens=0.9890  spec=0.8452
  R1: sens=0.7778  spec=0.9107
  R2: sens=0.5833  spec=0.9816
  R3A: sens=0.3333  spec=1.0000

── P2B  | PtMax  | Argmax  (baseline) ──
  AUROC=0.9396  Kappa=0.8007

## Comparison Table

In [6]:
rows = [
    ("P2B | Image  | Argmax (baseline)",  m_p2b_img),
    ("P2B | Image  | TTA + Argmax",       m_tta_img),
    ("P2B | PtMean | Argmax (best so far)",m_p2b_pt_mean),
    ("P2B | PtMean | TTA + Argmax",       m_tta_pt_mean),
    ("P2B | PtMax  | Argmax (baseline)",  m_p2b_pt_max),
    ("P2B | PtMax  | TTA + Argmax",       m_tta_pt_max),
]

cols = ["Configuration", "AUROC", "Kappa", "MacroSens", "R0", "R1", "R2", "R3A"]
hdr  = f"{cols[0]:<38} | {cols[1]:>6} | {cols[2]:>6} | {cols[3]:>9} | {cols[4]:>6} | {cols[5]:>6} | {cols[6]:>6} | {cols[7]:>6}"
print(hdr)
print("-" * len(hdr))

prev_group = None
for label, m in rows:
    group = label.split("|")[1].strip()
    if prev_group and group != prev_group:
        print()
    prev_group = group
    s = m["sensitivity"]
    print(f"{label:<38} | {m['auroc']:>6.4f} | {m['kappa']:>6.4f} | {m['macro_sens']:>9.4f} | {s[0]:>6.4f} | {s[1]:>6.4f} | {s[2]:>6.4f} | {s[3]:>6.4f}")

print()
# Deltas for PtMean (the recommended config)
b, t = m_p2b_pt_mean, m_tta_pt_mean
print("Delta (TTA vs baseline) for PtMean + Argmax:")
print(f"  AUROC      : {t['auroc']:.4f} vs {b['auroc']:.4f}  ({t['auroc']-b['auroc']:+.4f})")
print(f"  Kappa      : {t['kappa']:.4f} vs {b['kappa']:.4f}  ({t['kappa']-b['kappa']:+.4f})")
print(f"  MacroSens  : {t['macro_sens']:.4f} vs {b['macro_sens']:.4f}  ({t['macro_sens']-b['macro_sens']:+.4f})")
for i, cn in enumerate(CLASS_NAMES):
    delta = t['sensitivity'][i] - b['sensitivity'][i]
    print(f"  {cn} sens  : {t['sensitivity'][i]:.4f} vs {b['sensitivity'][i]:.4f}  ({delta:+.4f})")


Configuration                          |  AUROC |  Kappa | MacroSens |     R0 |     R1 |     R2 |    R3A
--------------------------------------------------------------------------------------------------------
P2B | Image  | Argmax (baseline)       | 0.9271 | 0.7671 |    0.6375 | 0.9463 | 0.7458 | 0.6078 | 0.2500
P2B | Image  | TTA + Argmax            | 0.9294 | 0.7606 |    0.6410 | 0.9361 | 0.7458 | 0.5490 | 0.3333

P2B | PtMean | Argmax (best so far)    | 0.9456 | 0.8212 |    0.6788 | 0.9890 | 0.8095 | 0.5833 | 0.3333
P2B | PtMean | TTA + Argmax            | 0.9450 | 0.8131 |    0.6709 | 0.9890 | 0.7778 | 0.5833 | 0.3333

P2B | PtMax  | Argmax (baseline)       | 0.9396 | 0.8007 |    0.6761 | 0.9780 | 0.8095 | 0.5833 | 0.3333
P2B | PtMax  | TTA + Argmax            | 0.9370 | 0.8220 |    0.6999 | 0.9780 | 0.7937 | 0.5833 | 0.4444

Delta (TTA vs baseline) for PtMean + Argmax:
  AUROC      : 0.9450 vs 0.9456  (-0.0007)
  Kappa      : 0.8131 vs 0.8212  (-0.0081)
  MacroSens  : 0.6709 vs 0